<a href="https://colab.research.google.com/github/albhoe/593Project/blob/main/tagproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate peft

In [ ]:
from google.colab import drive
import pandas as pd
import torch
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    BartForSequenceClassification,
    TrainingArguments,
    Trainer,
    get_scheduler
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel
import io
import re
import os
import gc
#import torch_xla.core.xla_model as xm
#import torch_xla.distributed.xla_multiprocessing as xmp
from torch.optim import AdamW
import numpy as np
from sklearn.metrics import accuracy_score

In [ ]:
drive.mount('/content/drive')

In [ ]:
file_path_drive = '/content/drive/MyDrive/593project/ao3_14400001-14500000.jsonl'
schema_path_drive = '/content/drive/MyDrive/593project'
save_path_drive = '/content/drive/MyDrive/593project/fine_tuned_bart_lora_classification_saved'

line_count = 0
df = pd.DataFrame()

with open(file_path_drive, 'r') as f:
    for line in f:
        if line_count >= 64:
            break
        line_count += 1
        # Use io.StringIO to pass the literal JSON string safely
        df = pd.concat([df, pd.read_json(io.StringIO(line), lines=True)], ignore_index=True)
metadata_df = pd.json_normalize(df['metadata'])
df = df.drop(columns=['metadata','id'])
df = pd.concat([df.reset_index(drop=True), metadata_df.reset_index(drop=True)], axis=1)
df = df.drop(columns=['published','words','Collections','Series','Fandoms','Archive Warnings','Relationships','Character','Categories'])

# Convert 'Category' column to one-hot encoding
df = pd.get_dummies(df, columns=['Category'], prefix='Category')

print("\nFirst 5 rows of data from Google Drive:")
display(df.head())

columns_list = df.columns.tolist()
output_filename = os.path.join(schema_path_drive, 'columns.txt')

with open(output_filename, 'w') as f:
  for col in columns_list:
    f.write(col + '\n')

Prepare the models

In [ ]:
import gc
import torch

# Delete large variables if any
# For example: del my_large_dataframe, my_large_model
# Then, run garbage collection
gc.collect()

# If using CUDA (GPU), clear the CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache cleared.")

print("RAM cleaning steps initiated.")

In [ ]:
num_ratings = len(df['Rating'].astype('category').cat.categories)
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

encoder = BartForSequenceClassification.from_pretrained('facebook/bart-base',num_labels=num_ratings)
classification_lora_config = LoraConfig(
    r=2,  # LoRA attention dimension
    lora_alpha=16, # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "v_proj"], # Target modules for BART's attention layers
    lora_dropout=0.1, # Dropout probability for LoRA layers
    bias="none", # Bias type for LoRA layers ('none', 'all', or 'lora_only')
    task_type="SEQ_CLS" # Specify the task type for sequence classification
)

Part 1: Rating Categorization.
This is the easiest part. Uses the BART encoder only.

In [ ]:
def rating_to_numerical(rating_series):
    # Define a mapping for common age ratings to numerical categories
    # You can customize these mappings as needed
    rating_map = {
        'Not Rated': 0,
        'General Audiences': 1,
        'Teen And Up Audiences': 2,
        'Mature': 3,
        'Explicit': 4
    }
    # Apply the mapping. Use -1 for ratings not found in the map
    return rating_series.map(rating_map).fillna(-1).astype(int)

def preprocess_function(examples):
    model_inputs = tokenizer(examples['text'], max_length=1024, truncation=True, padding="max_length")
    model_inputs["labels"] = examples["Rating"]
    return model_inputs

rating_df = df[['Rating','text']].copy(deep=True)
rating_df['Rating'] = rating_to_numerical(rating_df['Rating'])
#Although other categories may be useful to decrease loss, the intent of this project is to produce
#a model that classifies by text alone.
display(rating_df.head())

dataset = Dataset.from_pandas(rating_df)

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=['Rating','text'])

train_dataset = tokenized_dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)['train']
eval_dataset = tokenized_dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)['test']

display(train_dataset)
display(eval_dataset)

In [ ]:
!pip install --upgrade torchao

In [ ]:
bf16_enabled = False
fp16_enabled = False
"""try:
    # Check for TPU
    #if xm.xla_resource_manager().get_xla_supported_device_type() == 'TPU':
        device = xm.xla_device()
        print(f"Using TPU: {device}")
        bf16_enabled = True # bf16 is often preferred for TPUs
    #else:
        #raise Exception("Not a TPU runtime.")
#except Exception:
    # Fallback to GPU or CPU"""
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {device}")
    fp16_enabled = True # fp16 is often preferred for GPUs
else:
    device = torch.device("cpu")
    print(f"Using CPU: {device}")


training_args = TrainingArguments(
    warmup_steps=10,
    output_dir='./results_classification',
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    logging_dir='./logs_classification',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    bf16=bf16_enabled, # Changed to True for bfloat16 mixed precision on XLA device (TPU)
    fp16=bf16_enabled, # Ensure fp16 is False when bf16 is True
    report_to='none',
    learning_rate=1e-3
)

classification_peft_model = get_peft_model(encoder, classification_lora_config)
classification_peft_model.to(device)

optimizer = AdamW(classification_peft_model.parameters(), lr=training_args.learning_rate)

total_train_steps = int(len(train_dataset) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs)

# Define a learning rate scheduler
lr_scheduler = get_scheduler(
    name=training_args.lr_scheduler_type, # Defaults to 'linear'
    optimizer=optimizer,
    num_warmup_steps=training_args.warmup_steps,
    num_training_steps=total_train_steps,
)


trainer = Trainer(
    model=classification_peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    optimizers=(optimizer, lr_scheduler) # Pass the custom optimizer and scheduler
)

trainer.train()

os.makedirs(save_path_drive, exist_ok=True)
classification_peft_model.to('cpu').save_pretrained(save_path_drive)

In [ ]:
reverse_rating_map = {
    -1: 'Unknown',
    0: 'Not Rated',
    1: 'General Audiences',
    2: 'Teen And Up Audiences',
    3: 'Mature',
    4: 'Explicit'
}

predictions_output = trainer.predict(eval_dataset)
predicted_logits = predictions_output.predictions[0]
true_labels = predictions_output.label_ids
predicted_labels = np.argmax(predicted_logits, axis=1)

accuracy = accuracy_score(true_labels, predicted_labels)

print(f"Accuracy on the evaluation set: {accuracy * 100:.2f}%")

print("\nSample of True vs. Predicted Labels:")
print(f"{'Index':<5} {'True Label':<20} {'Predicted Label':<20}")
print(f"{'-----':<5} {'----------':<20} {'---------------':<20}")
for i in range(min(10, len(true_labels))):
    true_rating = reverse_rating_map.get(true_labels[i], f"Unknown ({true_labels[i]})")
    predicted_rating = reverse_rating_map.get(predicted_labels[i], f"Unknown ({predicted_labels[i]})")
    print(f"{i:<5} {true_rating:<20} {predicted_rating:<20}")

print(f"\nEvaluation metrics provided by Trainer: {predictions_output.metrics}")